# CRML Experiments

Builds the `:experiments` Gradle submodule (shadow JAR), starts a JVM via JPype, and exercises the CRML compiler pipeline directly from Python.

In [1]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent  # CRML repo root
from experiments.gradle_jvm import GradleJvm

## Build & start JVM

Runs `./gradlew experiments:shadowJar` and starts a JVM with the fat JAR on the classpath.  
The JVM can only be started once per kernel — restart the kernel to rebuild.

In [2]:
jvm = GradleJvm(
    project_path=ROOT,
    subproject="experiments",
    subproject_dir="submodules/experiments",
)
jvm.build()
jvm.start()

import jpype.imports

Running: /home/ubuntu/crml/vol/CRML/gradlew experiments:shadowJar  (cwd=/home/ubuntu/crml/vol/CRML)
> Task :language:generateGrammarSource

> Task :language:compileJava

> Task :util:generateGrammarSource NO-SOURCE
> Task :util:compileJava UP-TO-DATE


3 warnings



> Task :compiler:compileJava

> Task :compiler:processResources UP-TO-DATE
> Task :compiler:classes
> Task :compiler:jar UP-TO-DATE
> Task :experiments:compileJava NO-SOURCE
> Task :experiments:processResources UP-TO-DATE
> Task :experiments:classes UP-TO-DATE
> Task :language:processResources UP-TO-DATE
> Task :language:classes
> Task :language:jar
> Task :util:processResources NO-SOURCE
> Task :util:classes UP-TO-DATE
> Task :util:jar UP-TO-DATE


3 warnings


> Task :experiments:shadowJar

[Incubating] Problems report is available at: file:///home/ubuntu/crml/vol/CRML/build/reports/problems/problems-report.html

Deprecated Gradle features were used in this build, making it incompatible with Gradle 10.

You can use '--warning-mode all' to show the individual deprecation warnings and determine if they come from your own scripts or plugins.

For more on this, please refer to https://docs.gradle.org/9.1.0/userguide/command_line_interface.html#sec:command_line_warnings in the Gradle documentation.

BUILD SUCCESSFUL in 9s
11 actionable tasks: 5 executed, 6 up-to-date
Consider enabling configuration cache to speed up this build: https://docs.gradle.org/9.1.0/userguide/configuration_cache_enabling.html
Fat JAR: /home/ubuntu/crml/vol/CRML/submodules/experiments/build/libs/experiments-1.0-SNAPSHOT-all.jar
JVM started.


## Seed model validation

Parse the seed models from `experiments.tests.TESTS` and report any syntax errors.  
Seeds are the starting CRML skeletons that each LLM interaction builds upon.

In [3]:
#import jpype
from crml.language.util import Parser
from crml.compiler.omc import OMGenerator
from crml.util import IOUtil

from experiments.tests import TESTS

def parse_seed(name: str, seed: str) -> None:
    result = Parser().parse(seed)
    syntax = result.syntax()
    if syntax.hasErrors():
        errors = [str(e) for e in syntax.errors()]
        print(f"{name}: {len(errors)} error(s)")
        for e in errors:
            print(f"  {e}")
        print(result.toPrettyTree())
    else:
        print(f"{name}: OK")

In [4]:
# SRI domain
for key in TESTS.SRI.keys():
    parse_seed(f"SRI.{key}", TESTS.SRI[key]["seed"])

SRI.temp: OK
SRI.speed: OK


In [5]:
# Traffic light domain
for key in TESTS.trafic.keys():
    parse_seed(f"trafic.{key}", TESTS.trafic[key]["seed"])

trafic.t1: OK


In [6]:
# Pumping system domain
for key in TESTS.pumpsystem.keys():
    parse_seed(f"pumpsystem.{key}", TESTS.pumpsystem[key]["seed"])

pumpsystem.t1: 2 error(s)
  ERROR (line 17:52): no viable alternative at input '(ident ='
  ERROR (line 17:52): mismatched input '=' expecting {'{', IDENT}
definition 
    definition_type model
    id PumpingSystem
    is
    dependency flatten{
        id Units
        ,
        id FORM_L
        }union
    {
    element_def 
        class_def class
            id Pump
            is{
            class_var_def 
                var_def 
                    type 
                        builtin_type String
                    id ident
                    ;
            class_var_def 
                var_def 
                    type 
                        builtin_type Boolean
                    id isStarted
                    isexternal;
            class_var_def 
                var_def 
                    type 
                        builtin_type Real
                    id temperature
                    isexternal;
            };
    element_def 
        class_def class
       

In [7]:
jvm.shutdown()

JVM shut down.
